In [1]:
import pandas as pd
import numpy as np


In [2]:
# Training data
y_train = pd.read_csv("original/water_quality_training_dataset.csv")
x_train_All = pd.read_csv("Data/training_merged.csv")
print("Training Response Features Shape: ", y_train.shape)
print("Training Features Shape: ", x_train_All.shape)
landsat_only_train = pd.read_csv("original/landsat_features_training.csv")
print(landsat_only_train.shape)

# Validation data
submission_template = pd.read_csv("Data/submission_template.csv")
x_test_All = pd.read_csv("Data/validation_merged.csv")


print(landsat_only_train.head())

Training Response Features Shape:  (9319, 6)
Training Features Shape:  (8985, 86)
(9319, 9)
    Latitude  Longitude Sample Date      nir    green   swir16   swir22  \
0 -28.760833  17.730278  02-01-2011  11190.0  11426.0   7687.5   7645.0   
1 -26.861111  28.884722  03-01-2011  17658.5   9550.0  13746.5  10574.0   
2 -26.450000  28.085833  03-01-2011  15210.0  10720.0  17974.0  14201.0   
3 -27.671111  27.236944  03-01-2011  14887.0  10943.0  13522.0  11403.0   
4 -27.356667  27.286389  03-01-2011  16828.5   9502.5  12665.5   9643.0   

       NDMI     MNDWI  
0  0.185538  0.195595  
1  0.124566 -0.180134  
2 -0.083293 -0.252805  
3  0.048048 -0.105416  
4  0.141147 -0.142683  


In [3]:
test_concat = y_train.merge(
    landsat_only_train.reset_index(), 
    on=["Latitude", "Longitude"], 
    how="left"
)

In [4]:
print(y_train.shape)
y_train[["Longitude","Latitude"]].nunique()

(9319, 6)


Longitude    161
Latitude     161
dtype: int64

In [5]:
import sys, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0,'.')

from panel_model import (
    PanelWaterQualityModel,
    spatial_cv,
    compute_site_means,
    TIME_VARIANT_COLS,
    TARGETS
)
print("imports OK")

imports OK


In [6]:
train = pd.read_csv("Data/training_merged.csv")
val = pd.read_csv("Data/validation_merged.csv")
submission_template = pd.read_csv("Data/submission_template.csv")

print(f"Train: {train.shape} | {train.groupby(["Latitude","Longitude"]).ngroups} sites")
print(f"Validation: {val.shape} | {val.groupby(["Latitude","Longitude"]).ngroups} sites")

Train: (8985, 86) | 162 sites
Validation: (200, 83) | 24 sites


In [7]:
def add_geology_flags(df):
    def flags(name):
        n = str(name).lower()
        is_karoo = int(any(k in n for k in [
            'karoo','beaufort','ecca','dwyka','molteno','elliot','clarens',
            'vryheid','volksrust','tarkastad','fort brown','waterford',
            'prince albert','emakwezini','pietermaritzburg','dolerite',
        ]))
        is_cape = int(any(k in n for k in [
            'cape supergroup','witteberg','bokkeveld','table mountain',
            'nardouw','ceres subgroup','natal group',
        ]))
        return pd.Series({
            'is_karoo_supergroup': is_karoo,
            'is_cape_supergroup':  is_cape,
            'is_beaufort_group':   int(any(k in n for k in ['beaufort','tarkastad','adelaide','molteno','elliot','clarens','emakwezini'])),
            'is_ecca_group':       int(any(k in n for k in ['ecca','volksrust','vryheid','fort brown','waterford','prince albert','pietermaritzburg'])),
            'is_dwyka_group':      int('dwyka' in n),
            'is_karoo_dolerite':   int('dolerite' in n),
        })
    flags_df = df['macrostrat_name'].apply(flags)
    return pd.concat([df, flags_df], axis=1)

train = add_geology_flags(train)
val = add_geology_flags(val)

print("Geology flag coverage - train vs. val:")
for col in ['is_karoo_supergroup','is_cape_supergroup','is_beaufort_group',
            'is_ecca_group','is_dwyka_group','is_karoo_dolerite']:
    print(f"  {col:25s}: train={train[col].mean():.2f}  val={val[col].mean():.2f}")

Geology flag coverage - train vs. val:
  is_karoo_supergroup      : train=0.48  val=0.78
  is_cape_supergroup       : train=0.09  val=0.12
  is_beaufort_group        : train=0.16  val=0.61
  is_ecca_group            : train=0.19  val=0.01
  is_dwyka_group           : train=0.08  val=0.00
  is_karoo_dolerite        : train=0.04  val=0.16


In [8]:
for target in TARGETS:
    print(f"\n{'='*55}")
    print(f"  {target}")
    print('='*55)
    spatial_cv(train, target=target, n_splits=5)


  Total Alkalinity
  Fold 1: R²=0.3500  (n_test=1796)
  Fold 2: R²=0.1735  (n_test=1796)
  Fold 3: R²=0.1552  (n_test=1798)
  Fold 4: R²=0.2124  (n_test=1796)
  Fold 5: R²=0.2286  (n_test=1799)
  → Mean R² = 0.2239

  Electrical Conductance
  Fold 1: R²=0.2515  (n_test=1796)
  Fold 2: R²=0.2408  (n_test=1796)
  Fold 3: R²=0.2845  (n_test=1798)
  Fold 4: R²=0.2019  (n_test=1796)
  Fold 5: R²=0.2614  (n_test=1799)
  → Mean R² = 0.2480

  Dissolved Reactive Phosphorus
  Fold 1: R²=0.0956  (n_test=1796)
  Fold 2: R²=0.1883  (n_test=1796)
  Fold 3: R²=0.0252  (n_test=1798)
  Fold 4: R²=0.0620  (n_test=1796)
  Fold 5: R²=0.0255  (n_test=1799)
  → Mean R² = 0.0793


In [9]:
models = {}
for target in TARGETS:
    m = PanelWaterQualityModel(target=target)
    m.fit(train)
    models[target] = m
    print(f"Trained: {target}")

Trained: Total Alkalinity
Trained: Electrical Conductance
Trained: Dissolved Reactive Phosphorus


In [11]:
# Compute val site means from the val X features (no y used)
val_site_means = compute_site_means(val, TIME_VARIANT_COLS)

submission = submission_template.copy()
for target in TARGETS:
    preds = models[target].predict(val, site_means=val_site_means)
    submission[target] = np.clip(preds, 0, None)
    print(f"{target:40s}: mean={preds.mean():.1f}  min={preds.min():.1f}  max={preds.max():.1f}")

submission.to_csv("Data/submission.csv", index=False)
print("\nSaved → Data/submission.csv")
submission.head()

Total Alkalinity                        : mean=89.6  min=26.1  max=184.1
Electrical Conductance                  : mean=486.0  min=-27.4  max=849.5
Dissolved Reactive Phosphorus           : mean=17.0  min=-11.4  max=50.9

Saved → Data/submission.csv


,Latitude,Longitude,Sample Date,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus
0,-32.043333,27.822778,01-09-2014,98.558187,309.021944,2.178756
1,-33.329167,26.077500,16-09-2015,99.035157,575.187350,12.585600
2,-32.991639,27.640028,07-05-2015,92.935530,377.695660,14.846166
3,-34.096389,24.439167,07-02-2012,56.181351,538.575147,26.044121
4,-32.000556,28.581667,01-10-2014,78.337770,172.135725,2.433336


In [12]:
[c for c in train.columns if 'soil' in c.lower()]

['soil', 'soil_ph', 'soil_cec', 'soil_oc']